Ten skrypt implementuje system Retrieval-Augmented Generation (RAG) używający biblioteki CrewAI, który kieruje zapytania użytkownika do odpowiednich źródeł informacji poprzez zespół współpracujących agentów. System składa się z pięciu agentów. Całość tworzy pipeline, który przetwarza zapytanie użytkownika.

# Setup

In [1]:
!pip install -q crewai  crewai_tools langchain_community  sentence-transformers  langchain_huggingface

Ten kod to polecenie dla interpretera Pythona, które uruchamia instalację pakietów niezbędnych do działania pewnego oprogramowania. Używa menedżera pakietów `pip`, standardowego narzędzia w ekosystemie Pythona służącego do pobierania i instalacji bibliotek oraz ich zależności.

Flaga `-q` (quiet) sprawia, że proces instalacji przebiega bez wyświetlania szczegółowych komunikatów o postępie na ekranie – jest to przydatne, gdy chcemy uniknąć zaśmiecania wyjścia konsoli.

Następnie wymienione są nazwy pakietów do zainstalowania:

*   `crewai`: Główny pakiet, który stanowi podstawę aplikacji CrewAI - frameworku do tworzenia agentów AI współpracujących ze sobą.
*   `crewai_tools`: Dodatkowe narzędzia rozszerzające funkcjonalność CrewAI.
*   `langchain_community`: Zbiór modułów i integracji dla LangChain, popularnego frameworku do budowy aplikacji opartych na modelach językowych (LLM).  Zawiera wiele gotowych komponentów i połączeń z różnymi usługami.
*   `sentence-transformers`: Biblioteka służąca do generowania wektorowych reprezentacji tekstu (embeddings), które są wykorzystywane w wielu zadaniach przetwarzania języka naturalnego, takich jak wyszukiwanie semantyczne czy grupowanie dokumentów.
*   `langchain_huggingface`: Integracja LangChain z modelami językowymi udostępnianymi przez Hugging Face – platformę oferującą szeroki wybór modeli AI.

W skrócie, ten kod instaluje zestaw bibliotek Pythona potrzebnych do uruchomienia aplikacji wykorzystującej agentów AI (CrewAI) w połączeniu z frameworkiem LangChain i modelami językowymi z Hugging Face, a także narzędziami do przetwarzania tekstu.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from crewai.tools import tool
from crewai_tools import PDFSearchTool
from crewai import Crew, Task, Agent
import os
from google.colab import userdata

import requests

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:623: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


Ten kod ładuje moduły i klasy potrzebne do stworzenia aplikacji wykorzystującej agentów AI w środowisku programistycznym.

`from langchain_openai import ChatOpenAI`: Importuje funkcjonalność pozwalającą na komunikację z modelami językowymi OpenAI, takimi jak GPT-4, za pomocą interfejsu czatu.

`from langchain_community.tools.tavily_search import TavilySearchResults`: Importuje narzędzie umożliwiające agentom wyszukiwanie informacji w internecie przy użyciu wyszukiwarki Tavily.

`from crewai.tools import tool`: Importuje dekorator, który pozwala na definiowanie własnych narzędzi dla agentów AI.

`from crewai_tools import PDFSearchTool`: Importuje narzędzie umożliwiające agentom przeszukiwanie zawartości plików PDF.

`from crewai import Crew, Task, Agent`: Importuje podstawowe klasy frameworku CrewAI: `Crew` reprezentuje grupę współpracujących agentów, `Task` definiuje zadanie do wykonania przez agenta lub zespół, a `Agent` to pojedynczy agent AI.

`import os`: Importuje moduł umożliwiający interakcję z systemem operacyjnym, na przykład odczytywanie zmiennych środowiskowych.

`from google.colab import userdata`: Importuje funkcjonalność pozwalającą na dostęp do danych użytkownika przechowywanych w środowisku Google Colab, takich jak klucze API.

`import requests`: Importuje bibliotekę umożliwiającą wysyłanie żądań HTTP, co pozwala agentom pobierać dane z internetu lub komunikować się z innymi usługami sieciowymi.

In [4]:
class CFG:
    model = 'gpt-4o-mini'
    emb_model = "BAAI/bge-small-en-v1.5"
    temperature=0.1
    max_tokens=1000
    paper = "https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf"

Ten kod definiuje klasę o nazwie `CFG`, która służy do przechowywania konfiguracji aplikacji. Klasa ta zawiera stałe wartości, które kontrolują działanie programu.

*   `model = 'gpt-4o-mini'`: Określa model językowy OpenAI, który będzie używany przez agentów AI. W tym przypadku jest to `gpt-4o-mini`.
*   `emb_model = "BAAI/bge-small-en-v1.5"`: Definiuje model do generowania osadzeń (embeddings) tekstu. Osadzenia reprezentują tekst w postaci wektorów liczb, co umożliwia wykonywanie operacji semantycznych na tekście. Używany jest model `BAAI/bge-small-en-v1.5`.
*   `temperature=0.1`: Kontroluje losowość generowanego tekstu przez model językowy. Niższa wartość (np. 0.1) powoduje, że tekst jest bardziej deterministyczny i przewidywalny, a wyższa wartość (np. 0.7) wprowadza większą różnorodność i kreatywność.
*   `max_tokens=1000`: Określa maksymalną liczbę tokenów (słów lub części słów), które model językowy może wygenerować w odpowiedzi. Ograniczenie to pomaga kontrolować długość generowanego tekstu i koszty związane z korzystaniem z modelu.
*   `paper = "https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf"`: Przechowuje adres URL do pliku PDF z artykułem naukowym, który będzie analizowany przez agentów AI.

In [8]:
os.environ["OPENAI_API_KEY"] = userdata.get('openaivision')
os.environ['TAVILY_API_KEY'] = userdata.get('tavily')

Ten kod ustawia zmienne środowiskowe, które są niezbędne do działania aplikacji wykorzystującej API OpenAI i Tavily.

*   `os.environ["OPENAI_API_KEY"] = userdata.get('openaivision')`: Pobiera klucz API OpenAI z danych użytkownika przechowywanych w Google Colab (pod nazwą 'openaivision') i ustawia go jako wartość zmiennej środowiskowej `OPENAI_API_KEY`.  Klucz API jest wymagany do autoryzacji żądań wysyłanych do serwerów OpenAI.
*   `os.environ['TAVILY_API_KEY'] = userdata.get('tavily')`: Pobiera klucz API Tavily z danych użytkownika w Google Colab (pod nazwą 'tavily') i ustawia go jako wartość zmiennej środowiskowej `TAVILY_API_KEY`. Klucz API jest wymagany do autoryzacji żądań wysyłanych do serwerów Tavily, umożliwiając korzystanie z ich usługi wyszukiwania.

Ustawienie kluczy API w zmiennych środowiskowych jest bezpieczniejszym sposobem zarządzania poufnymi danymi niż umieszczanie ich bezpośrednio w kodzie źródłowym.

# Model

In [9]:
llm = ChatOpenAI(
    model_name = CFG.model,
    temperature = CFG.temperature,
    max_tokens = CFG.max_tokens
)

Ten kod tworzy instancję klasy `ChatOpenAI`, która reprezentuje połączenie z modelem językowym OpenAI i konfiguruje jego parametry.

*   `llm = ChatOpenAI(...)`: Tworzy nowy obiekt o nazwie `llm` będący instancją klasy `ChatOpenAI`. Ten obiekt będzie używany do komunikacji z modelem językowym OpenAI.
*   `model_name = CFG.model`: Ustawia nazwę modelu językowego, który ma być używany. Wartość jest pobierana z atrybutu `model` klasy `CFG`, czyli w tym przypadku `gpt-4o-mini`.
*   `temperature = CFG.temperature`: Ustawia temperaturę generowania tekstu przez model. Wartość jest pobierana z atrybutu `temperature` klasy `CFG`, czyli 0.1. Niższa temperatura oznacza bardziej deterministyczne odpowiedzi.
*   `max_tokens = CFG.max_tokens`: Ustawia maksymalną liczbę tokenów, które model może wygenerować w jednej odpowiedzi. Wartość jest pobierana z atrybutu `max_tokens` klasy `CFG`, czyli 1000.

W rezultacie powstaje obiekt `llm`, który jest skonfigurowany do korzystania z modelu językowego `gpt-4o-mini` o temperaturze 0.1 i maksymalnej długości odpowiedzi wynoszącej 1000 tokenów.

# Narzędzia

In [10]:
response = requests.get(CFG.paper)
with open("attenstion_is_all_you_need.pdf", "wb") as file:
    file.write(response.content)

Ten kod pobiera plik PDF z podanego adresu URL i zapisuje go lokalnie na dysku.

*   `response = requests.get(CFG.paper)`: Wykorzystuje bibliotekę `requests` do wysłania żądania HTTP GET na adres URL przechowywany w zmiennej `CFG.paper`. Adres ten wskazuje na plik PDF z artykułem naukowym "Attention is All You Need". Odpowiedź serwera (zawierająca zawartość pliku PDF) jest zapisywana w obiekcie `response`.
*   `with open("attenstion_is_all_you_need.pdf", "wb") as file:`: Otwiera plik o nazwie "attenstion\_is\_all\_you\_need.pdf" w trybie zapisu binarnego ("wb"). Konstrukcja `with open(...) as file:` zapewnia, że plik zostanie automatycznie zamknięty po zakończeniu bloku kodu, nawet jeśli wystąpią błędy.
*   `file.write(response.content)`: Zapisuje zawartość pobranego pliku PDF (przechowywaną w `response.content`) do otwartego pliku "attenstion\_is\_all\_you\_need.pdf".

W efekcie tego kodu, plik PDF z artykułem "Attention is All You Need" zostaje pobrany z internetu i zapisany na dysku pod nazwą "attenstion\_is\_all\_you\_need.pdf".

In [12]:
rag_tool = PDFSearchTool(pdf='attenstion_is_all_you_need.pdf',
    config=dict(
        llm=dict(
            provider="openai",
            config=dict(
                model = CFG.model,
                temperature = CFG.temperature,
            ),
        ),
        embedder=dict(
            provider="huggingface",
            config=dict(
                model = CFG.emb_model,
            ),
        ),
    )
)

/usr/local/lib/python3.11/dist-packages/embedchain/embedder/huggingface.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=self.config.model, model_kwargs=self.config.model_kwargs)


Ten kod tworzy instancję narzędzia `PDFSearchTool`, które umożliwia wyszukiwanie informacji w pliku PDF z wykorzystaniem technik RAG (Retrieval-Augmented Generation).

*   `rag_tool = PDFSearchTool(...)`: Tworzy nowy obiekt o nazwie `rag_tool` będący instancją klasy `PDFSearchTool`.
*   `pdf='attenstion_is_all_you_need.pdf'`: Określa ścieżkę do pliku PDF, w którym ma być przeprowadzane wyszukiwanie. W tym przypadku jest to "attenstion\_is\_all\_you\_need.pdf", który został wcześniej pobrany i zapisany na dysku.
*   `config=dict(...)`: Przekazuje konfigurację narzędzia w postaci słownika. Konfiguracja ta definiuje, jak ma być przeprowadzane wyszukiwanie i generowanie odpowiedzi.
    *   `llm=dict(...)`: Definiuje konfigurację modelu językowego (LLM) używanego do generowania odpowiedzi na podstawie pobranych fragmentów tekstu z PDF.
        *   `provider="openai"`: Określa, że dostawcą LLM jest OpenAI.
        *   `config=dict(...)`: Zawiera dodatkowe ustawienia dla modelu OpenAI.
            *   `model = CFG.model`: Ustawia nazwę modelu językowego na wartość zdefiniowaną w `CFG.model`, czyli `gpt-4o-mini`.
            *   `temperature = CFG.temperature`: Ustawia temperaturę generowania tekstu na wartość zdefiniowaną w `CFG.temperature`, czyli 0.1.
    *   `embedder=dict(...)`: Definiuje konfigurację modelu do generowania osadzeń (embeddings) używanego do reprezentacji fragmentów tekstu z PDF.
        *   `provider="huggingface"`: Określa, że dostawcą modelu embeddings jest Hugging Face.
        *   `config=dict(...)`: Zawiera dodatkowe ustawienia dla modelu embeddings.
            *   `model = CFG.emb_model`: Ustawia nazwę modelu embeddings na wartość zdefiniowaną w `CFG.emb_model`, czyli `BAAI/bge-small-en-v1.5`.

W rezultacie powstaje obiekt `rag_tool`, który jest skonfigurowany do przeszukiwania pliku PDF "attenstion\_is\_all\_you\_need.pdf" z wykorzystaniem modelu językowego `gpt-4o-mini` i modelu embeddings `BAAI/bge-small-en-v1.5`. Narzędzie to będzie mogło odpowiadać na pytania dotyczące zawartości pliku PDF, pobierając odpowiednie fragmenty tekstu i generując odpowiedzi przy użyciu LLM.

In [13]:
# safety check

rag_tool.run("How did self-attention mechanism evolve in large language models?")

Using Tool: Search a PDF's content


/usr/local/lib/python3.11/dist-packages/chromadb/types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x


'Relevant Content:\nmakes it more difﬁcult to learn dependencies between distant positions [ 11]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 22, 23, 19]. End-to-end memory networks are based on a recurrent attention mechanism instead of sequence- aligned recurrence and have been shown to perform well on simple-language question answering and language modeling tasks [28]. To the best of our knowledge, however, t

Ten kod uruchamia narzędzie `rag_tool` (PDFSearchTool) w celu znalezienia odpowiedzi na zadane pytanie dotyczące mechanizmu samoatencji w dużych modelach językowych.

*   `rag_tool.run("How did self-attention mechanism evolve in large language models?")`: Wywołuje metodę `run()` narzędzia `rag_tool`, przekazując jako argument zapytanie "How did self-attention mechanism evolve in large language models?". Metoda ta przeszukuje plik PDF ("attenstion\_is\_all\_you\_need.pdf") w poszukiwaniu fragmentów tekstu związanych z tym pytaniem, a następnie wykorzystuje model językowy (LLM) do wygenerowania odpowiedzi na podstawie znalezionych informacji.

W efekcie tego kodu narzędzie `rag_tool` przeszuka plik PDF i zwróci odpowiedź na pytanie dotyczące ewolucji mechanizmu samoatencji w dużych modelach językowych, opierając się na zawartości dokumentu.

In [14]:
web_search_tool = TavilySearchResults(k=3)
web_search_tool.run("What is self-attention mechansim in large language models?")

[{'title': 'Exploring Attention Mechanisms in Large Language Models - Medium',
  'url': 'https://medium.com/data-science-community-srm/exploring-attention-mechanisms-in-large-language-models-33549ae2c5b8',
  'content': 'Imagine each word in a sentence as a friend in a group chat. The self-attention mechanism lets the model look at each friend’s message, and based on what they said, decide how much attention to give to each one. This helps the model figure out the context and meaning of the words, making them really good at understanding language. [...] In the Transformer architecture, the self-attention mechanism plays a crucial role by helping the model decide how important each word or token is in a sequence. It does this by giving attention scores to words based on how relevant they are to others in the sequence. This way, the model can focus on the most contextually important information. [...] Now, self-attention is a feature in their design that helps them understand words and la

Ten kod wykorzystuje narzędzie `web_search_tool` do wyszukania informacji w internecie na temat mechanizmu samoatencji w dużych modelach językowych.

*   `web_search_tool = TavilySearchResults(k=3)`: Tworzy instancję klasy `TavilySearchResults`, która reprezentuje narzędzie do wyszukiwania w internecie przy użyciu API Tavily. Parametr `k=3` określa, że narzędzie ma zwrócić 3 wyniki wyszukiwania.
*   `web_search_tool.run("What is self-attention mechansim in large language models?")`: Wywołuje metodę `run()` narzędzia `web_search_tool`, przekazując jako argument zapytanie "What is self-attention mechansim in large language models?". Metoda ta wysyła zapytanie do API Tavily i pobiera 3 najlepsze wyniki wyszukiwania związane z tym tematem.

W efekcie tego kodu narzędzie `web_search_tool` przeszuka internet przy użyciu Tavily i zwróci 3 linki do stron internetowych zawierających informacje na temat mechanizmu samoatencji w dużych modelach językowych.

In [15]:
@tool
def router_tool(question):
  """Router Function"""
  if 'self-attention' in question:
    return 'vectorstore'
  else:
    return 'web_search'

Ten kod definiuje niestandardowe narzędzie o nazwie `router_tool`, które służy do kierowania zapytań do odpowiednich źródeł informacji.

*   `@tool`: Jest to dekorator, który oznacza funkcję jako narzędzie dostępne dla agentów AI w frameworku CrewAI.
*   `def router_tool(question):`: Definiuje funkcję o nazwie `router_tool`, która przyjmuje jeden argument: `question` (zapytanie).
*   `"""Router Function"""`: Jest to docstring, który opisuje przeznaczenie funkcji – jest to funkcja routingu.
*   `if 'self-attention' in question:`: Sprawdza, czy w zapytaniu występuje fraza "self-attention".
    *   `return 'vectorstore'`: Jeśli fraza "self-attention" znajduje się w zapytaniu, funkcja zwraca wartość `'vectorstore'`, co sugeruje, że zapytanie powinno być skierowane do wektora bazy danych (zawierającej informacje z pliku PDF).
*   `else:`: Jeśli fraza "self-attention" nie występuje w zapytaniu.
    *   `return 'web_search'`: Funkcja zwraca wartość `'web_search'`, co sugeruje, że zapytanie powinno być skierowane do wyszukiwarki internetowej.

W skrócie, funkcja `router_tool` analizuje treść zapytania i na podstawie obecności frazy "self-attention" decyduje, czy odpowiedź powinna być szukana w lokalnej bazie wektorowej (z pliku PDF), czy w internecie za pomocą wyszukiwarki.

# Agenci

In [16]:
Router_Agent = Agent(
  role='Router',
  goal='Route user question to a vectorstore or web search',
  backstory=(
    "You are an expert at routing a user question to a vectorstore or web search."
    "Use the vectorstore for questions on concept related to Retrieval-Augmented Generation."
    "You do not need to be stringent with the keywords in the question related to these topics. Otherwise, use web-search."
  ),
  verbose=True,
  allow_delegation=False,
  llm=llm,
)

Ten kod tworzy agenta o nazwie `Router_Agent`, którego zadaniem jest kierowanie zapytań użytkownika do odpowiedniego źródła informacji – bazy wektorowej (vectorstore) lub wyszukiwarki internetowej.

*   `Router_Agent = Agent(...)`: Tworzy nowy obiekt o nazwie `Router_Agent` będący instancją klasy `Agent`.
*   `role='Router'`: Definiuje rolę agenta jako "Router", czyli kierownik zapytań.
*   `goal='Route user question to a vectorstore or web search'`: Określa cel agenta – przekierowanie pytania użytkownika do bazy wektorowej lub wyszukiwarki internetowej.
*   `backstory=(...)`: Definiuje historię (opis) agenta, która pomaga mu w podejmowaniu decyzji. Historia ta informuje agenta, że jest ekspertem w kierowaniu zapytań i powinien używać bazy wektorowej do pytań związanych z Retrieval-Augmented Generation (RAG), a w przeciwnym razie korzystać z wyszukiwarki internetowej.
*   `verbose=True`: Włącza tryb szczegółowy, co oznacza, że agent będzie wyświetlał informacje o swoich działaniach i decyzjach.
*   `allow_delegation=False`: Wyłącza możliwość delegowania zadań przez agenta do innych agentów. Oznacza to, że `Router_Agent` samodzielnie podejmuje decyzję o kierowaniu zapytania.
*   `llm=llm`: Przypisuje model językowy (`llm`), który został wcześniej skonfigurowany (w kodzie `llm = ChatOpenAI(...)`), do agenta. Model ten będzie używany przez agenta do analizy zapytań i podejmowania decyzji o routingu.

W rezultacie powstaje agent `Router_Agent`, który jest gotowy do odbierania pytań od użytkownika, analizowania ich treści i kierowania ich do odpowiedniego źródła informacji na podstawie swojej roli, celu i historii.

In [17]:
Retriever_Agent = Agent(
role="Retriever",
goal="Use the information retrieved from the vectorstore to answer the question",
backstory=(
    "You are an assistant for question-answering tasks."
    "Use the information present in the retrieved context to answer the question."
    "You have to provide a clear concise answer."
),
verbose=True,
allow_delegation=False,
llm=llm,
)

Ten kod tworzy agenta o nazwie `Retriever_Agent`, którego zadaniem jest odpowiadanie na pytania użytkownika w oparciu o informacje pobrane z bazy wektorowej.

*   `Retriever_Agent = Agent(...)`: Tworzy nowy obiekt o nazwie `Retriever_Agent` będący instancją klasy `Agent`.
*   `role="Retriever"`: Definiuje rolę agenta jako "Retriever", czyli pobieracz informacji.
*   `goal="Use the information retrieved from the vectorstore to answer the question"`: Określa cel agenta – wykorzystanie informacji pobranych z bazy wektorowej do udzielenia odpowiedzi na pytanie.
*   `backstory=(...)`: Definiuje historię (opis) agenta, która pomaga mu w podejmowaniu decyzji. Historia ta informuje agenta, że jest asystentem do odpowiadania na pytania i powinien używać informacji z pobranego kontekstu, aby udzielać jasnych i zwięzłych odpowiedzi.
*   `verbose=True`: Włącza tryb szczegółowy, co oznacza, że agent będzie wyświetlał informacje o swoich działaniach i decyzjach.
*   `allow_delegation=False`: Wyłącza możliwość delegowania zadań przez agenta do innych agentów. Oznacza to, że `Retriever_Agent` samodzielnie odpowiada na pytania w oparciu o pobrane dane.
*   `llm=llm`: Przypisuje model językowy (`llm`), który został wcześniej skonfigurowany (w kodzie `llm = ChatOpenAI(...)`), do agenta. Model ten będzie używany przez agenta do generowania odpowiedzi na pytania.

W rezultacie powstaje agent `Retriever_Agent`, który jest gotowy do odbierania pytań i kontekstu z bazy wektorowej, analizowania tych informacji i udzielania jasnych i zwięzłych odpowiedzi.

In [18]:
Grader_agent =  Agent(
  role='Answer Grader',
  goal='Filter out erroneous retrievals',
  backstory=(
    "You are a grader assessing relevance of a retrieved document to a user question."
    "If the document contains keywords related to the user question, grade it as relevant."
    "It does not need to be a stringent test.You have to make sure that the answer is relevant to the question."
  ),
  verbose=True,
  allow_delegation=False,
  llm=llm,
)

Ten kod tworzy agenta o nazwie `Grader_agent`, którego zadaniem jest ocena trafności pobranych dokumentów w odniesieniu do pytania użytkownika.

*   `Grader_agent = Agent(...)`: Tworzy nowy obiekt o nazwie `Grader_agent` będący instancją klasy `Agent`.
*   `role='Answer Grader'`: Definiuje rolę agenta jako "Answer Grader", czyli oceniający odpowiedzi.
*   `goal='Filter out erroneous retrievals'`: Określa cel agenta – eliminacja błędnych pobrań (nieadekwatnych dokumentów).
*   `backstory=(...)`: Definiuje historię (opis) agenta, która pomaga mu w podejmowaniu decyzji. Historia ta informuje agenta, że jest oceniającym trafność dokumentu w odniesieniu do pytania użytkownika i powinien uznać go za trafny, jeśli zawiera słowa kluczowe związane z pytaniem. Podkreśla się, że test nie musi być rygorystyczny, a celem jest zapewnienie relewantności odpowiedzi na pytanie.
*   `verbose=True`: Włącza tryb szczegółowy, co oznacza, że agent będzie wyświetlał informacje o swoich działaniach i decyzjach.
*   `allow_delegation=False`: Wyłącza możliwość delegowania zadań przez agenta do innych agentów. Oznacza to, że `Grader_agent` samodzielnie ocenia trafność dokumentów.
*   `llm=llm`: Przypisuje model językowy (`llm`), który został wcześniej skonfigurowany (w kodzie `llm = ChatOpenAI(...)`), do agenta. Model ten będzie używany przez agenta do analizy treści dokumentu i pytania oraz podejmowania decyzji o trafności.


In [19]:
hallucination_grader = Agent(
    role="Hallucination Grader",
    goal="Filter out hallucination",
    backstory=(
        "You are a hallucination grader assessing whether an answer is grounded in / supported by a set of facts."
        "Make sure you meticulously review the answer and check if the response provided is in alignmnet with the question asked"
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

Ten kod tworzy agenta o nazwie `hallucination_grader`, którego zadaniem jest wykrywanie i eliminowanie halucynacji w odpowiedziach generowanych przez model językowy.

*   `hallucination_grader = Agent(...)`: Tworzy nowy obiekt o nazmie `hallucination_grader` będący instancją klasy `Agent`.
*   `role="Hallucination Grader"`: Definiuje rolę agenta jako "Hallucination Grader", czyli oceniający halucynacje.
*   `goal="Filter out hallucination"`: Określa cel agenta – eliminacja halucynacji (nieprawdziwych lub niezgodnych z faktami informacji) w odpowiedziach.
*   `backstory=(...)`: Definiuje historię (opis) agenta, która pomaga mu w podejmowaniu decyzji. Historia ta informuje agenta, że jest oceniającym halucynacje i powinien dokładnie sprawdzać, czy odpowiedź jest oparta na faktach i zgodna z pytaniem.
*   `verbose=True`: Włącza tryb szczegółowy, co oznacza, że agent będzie wyświetlał informacje o swoich działaniach i decyzjach.
*   `allow_delegation=False`: Wyłącza możliwość delegowania zadań przez agenta do innych agentów. Oznacza to, że `hallucination_grader` samodzielnie ocenia obecność halucynacji w odpowiedziach.
*   `llm=llm`: Przypisuje model językowy (`llm`), który został wcześniej skonfigurowany (w kodzie `llm = ChatOpenAI(...)`), do agenta. Model ten będzie używany przez agenta do analizy treści odpowiedzi i oceny jej zgodności z faktami.

W rezultacie powstaje agent `hallucination_grader`, który jest gotowy do odbierania odpowiedzi i oceniania, czy zawierają one halucynacje, czyli informacje niezgodne z prawdą lub niepoparte dowodami.

In [20]:
answer_grader = Agent(
    role="Answer Grader",
    goal="Filter out hallucination from the answer.",
    backstory=(
        "You are a grader assessing whether an answer is useful to resolve a question."
        "Make sure you meticulously review the answer and check if it makes sense for the question asked"
        "If the answer is relevant generate a clear and concise response."
        "If the answer gnerated is not relevant then perform a websearch using 'web_search_tool'"
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

Ten kod tworzy agenta o nazwie `answer_grader`, którego zadaniem jest ocena jakości odpowiedzi i w razie potrzeby uzupełnienie jej informacjami z wyszukiwarki internetowej.

*   `answer_grader = Agent(...)`: Tworzy nowy obiekt o nazwie `answer_grader` będący instancją klasy `Agent`.
*   `role="Answer Grader"`: Definiuje rolę agenta jako "Answer Grader", czyli oceniający odpowiedzi.
*   `goal="Filter out hallucination from the answer."`: Określa cel agenta – eliminacja halucynacji z odpowiedzi, czyli zapewnienie jej zgodności z faktami.
*   `backstory=(...)`: Definiuje historię (opis) agenta, która pomaga mu w podejmowaniu decyzji. Historia ta informuje agenta, że jest oceniającym przydatność odpowiedzi i powinien dokładnie sprawdzić, czy ma ona sens w kontekście pytania. Jeśli odpowiedź jest trafna, ma wygenerować jasną i zwięzłą wersję. Jeżeli odpowiedź nie jest adekwatna, ma przeprowadzić wyszukiwanie w internecie za pomocą narzędzia `web_search_tool`.
*   `verbose=True`: Włącza tryb szczegółowy, co oznacza, że agent będzie wyświetlał informacje o swoich działaniach i decyzjach.
*   `allow_delegation=False`: Wyłącza możliwość delegowania zadań przez agenta do innych agentów. Oznacza to, że `answer_grader` samodzielnie ocenia odpowiedzi i w razie potrzeby wyszukuje dodatkowe informacje.
*   `llm=llm`: Przypisuje model językowy (`llm`), który został wcześniej skonfigurowany (w kodzie `llm = ChatOpenAI(...)`), do agenta. Model ten będzie używany przez agenta do analizy treści odpowiedzi i podejmowania decyzji o jej jakości oraz ewentualnym wyszukiwaniu dodatkowych informacji.

W rezultacie powstaje agent `answer_grader`, który jest gotowy do odbierania odpowiedzi, oceniania ich trafności i przydatności, a w razie potrzeby uzupełniania ich informacjami z internetu za pomocą narzędzia `web_search_tool`.

# Zadania

In [21]:
router_task = Task(
    description=("Analyse the keywords in the question {question}"
    "Based on the keywords decide whether it is eligible for a vectorstore search or a web search."
    "Return a single word 'vectorstore' if it is eligible for vectorstore search."
    "Return a single word 'websearch' if it is eligible for web search."
    "Do not provide any other premable or explaination."
    ),
    expected_output=("Give a binary choice 'websearch' or 'vectorstore' based on the question"
    "Do not provide any other premable or explaination."),
    agent=Router_Agent,
    tools=[router_tool],
)

Ten kod definiuje zadanie o nazwie `router_task`, które ma na celu przekierowanie pytania użytkownika do odpowiedniego źródła informacji – bazy wektorowej lub wyszukiwarki internetowej.

*   `router_task = Task(...)`: Tworzy nowy obiekt o nazwie `router_task` będący instancją klasy `Task`.
*   `description=(...)`: Definiuje opis zadania, który instruuje agenta, jak ma wykonać to zadanie. Opis ten nakazuje agentowi analizę słów kluczowych w pytaniu i podjęcie decyzji, czy pytanie jest odpowiednie do wyszukiwania w bazie wektorowej (vectorstore) czy w internecie (web search). Agent ma zwrócić pojedyncze słowo "vectorstore" lub "websearch", bez dodatkowych wyjaśnień.
*   `expected_output=(...)`: Definiuje oczekiwany wynik zadania, czyli pojedyncze słowo "websearch" lub "vectorstore", bez żadnych wstępów ani objaśnień.
*   `agent=Router_Agent`: Przypisuje agenta `Router_Agent` do tego zadania. Agent ten będzie odpowiedzialny za wykonanie zadania zgodnie z podanym opisem.
*   `tools=[router_tool]`: Określa listę narzędzi, które są dostępne dla agenta podczas wykonywania zadania. W tym przypadku jedynym dostępnym narzędziem jest `router_tool`, czyli funkcja routingu zdefiniowana wcześniej w kodzie.

zawierającej informacje z pliku PDF).
*   `else:`: Jeśli fraza "self-attention" nie występuje w zapytaniu.
    *   `return 'web_search'`: Funkcja zwraca wartość `'web_search'`, co sugeruje, że zapytanie powinno być skierowane do wyszukiwarki internetowej.

W skrócie, funkcja `router_tool` analizuje treść zapytania i na podstawie

In [22]:
retriever_task = Task(
    description=("Based on the response from the router task extract information for the question {question} with the help of the respective tool."
    "Use the web_serach_tool to retrieve information from the web in case the router task output is 'websearch'."
    "Use the rag_tool to retrieve information from the vectorstore in case the router task output is 'vectorstore'."
    ),
    expected_output=("You should analyse the output of the 'router_task'"
    "If the response is 'websearch' then use the web_search_tool to retrieve information from the web."
    "If the response is 'vectorstore' then use the rag_tool to retrieve information from the vectorstore."
    "Return a claer and consise text as response."),
    agent=Retriever_Agent,
    context=[router_task],
)

Ten kod definiuje zadanie o nazwie `retriever_task`, którego celem jest pobranie informacji z odpowiedniego źródła (bazy wektorowej lub internetu) na podstawie decyzji podjętej przez agenta routingu.

*   `retriever_task = Task(...)`: Tworzy nowy obiekt o nazwie `retriever_task` będący instancją klasy `Task`.
*   `description=(...)`: Definiuje opis zadania, który instruuje agenta, jak ma wykonać to zadanie. Opis ten nakazuje agentowi pobranie informacji na podstawie wyniku z zadania routingu (`router_task`). Jeśli wynik jest "websearch", agent ma użyć narzędzia `web_serach_tool` do wyszukiwania w internecie. Jeśli wynik jest "vectorstore", agent ma użyć narzędzia `rag_tool` do przeszukiwania bazy wektorowej.
*   `expected_output=(...)`: Definiuje oczekiwany wynik zadania, czyli analizę wyniku z zadania routingu i wykorzystanie odpowiedniego narzędzia (web\_search\_tool lub rag\_tool) do pobrania informacji. Agent ma zwrócić jasną i zwięzłą treść jako odpowiedź.
*   `agent=Retriever_Agent`: Przypisuje agenta `Retriever_Agent` do tego zadania. Agent ten będzie odpowiedzialny za wykonanie zadania zgodnie z podanym opisem.
*   `context=[router_task]`: Określa kontekst zadania, czyli listę zadań, których wyniki są dostępne dla agenta podczas wykonywania tego zadania. W tym przypadku jedynym elementem w kontekście jest `router_task`, co oznacza, że agent `Retriever_Agent` będzie miał dostęp do wyniku wykonania zadania routingu.

W rezultacie powstaje zadanie `retriever_task`, które ma na celu pobranie informacji z odpowiedniego źródła na podstawie decyzji podjętej przez agenta routingu i wykorzystuje agenta `Retriever_Agent` oraz narzędzia `web_serach_tool` lub `rag_tool`.

In [23]:
grader_task = Task(
    description=("Based on the response from the retriever task for the quetion {question} evaluate whether the retrieved content is relevant to the question."
    ),
    expected_output=("Binary score 'yes' or 'no' score to indicate whether the document is relevant to the question"
    "You must answer 'yes' if the response from the 'retriever_task' is in alignment with the question asked."
    "You must answer 'no' if the response from the 'retriever_task' is not in alignment with the question asked."
    "Do not provide any preamble or explanations except for 'yes' or 'no'."),
    agent=Grader_agent,
    context=[retriever_task],
)

Ten kod definiuje zadanie o nazwie `grader_task`, którego celem jest ocena trafności odpowiedzi pobranej przez agenta retrievera w odniesieniu do zadanego pytania.

*   `grader_task = Task(...)`: Tworzy nowy obiekt o nazwie `grader_task` będący instancją klasy `Task`.
*   `description=(...)`: Definiuje opis zadania, który instruuje agenta, aby ocenił, czy treść pobrana przez agenta retrievera jest istotna dla pytania.
*   `expected_output=(...)`: Definiuje oczekiwany wynik zadania, czyli binarną ocenę "yes" (tak) lub "no" (nie), wskazującą na trafność dokumentu w odniesieniu do pytania. Agent ma odpowiedzieć "yes", jeśli odpowiedź z `retriever_task` jest zgodna z pytaniem, a "no" w przeciwnym przypadku. Wynik ma być ograniczony wyłącznie do słów "yes" lub "no".
*   `agent=Grader_agent`: Przypisuje agenta `Grader_agent` do tego zadania. Agent ten będzie odpowiedzialny za wykonanie zadania zgodnie z podanym opisem.
*   `context=[retriever_task]`: Określa kontekst zadania, czyli listę zadań, których wyniki są dostępne dla agenta podczas wykonywania tego zadania. W tym przypadku jedynym elementem w kontekście jest `retriever_task`, co oznacza, że agent `Grader_agent` będzie miał dostęp do wyniku wykonania zadania retrievera.

W rezultacie powstaje zadanie `grader_task`, które ma na celu ocenę trafności odpowiedzi pobranej przez agenta retrievera i wykorzystuje agenta `Grader_agent`.

In [24]:
hallucination_task = Task(
    description=("Based on the response from the grader task for the quetion {question} evaluate whether the answer is grounded in / supported by a set of facts."),
    expected_output=("Binary score 'yes' or 'no' score to indicate whether the answer is sync with the question asked"
    "Respond 'yes' if the answer is in useful and contains fact about the question asked."
    "Respond 'no' if the answer is not useful and does not contains fact about the question asked."
    "Do not provide any preamble or explanations except for 'yes' or 'no'."),
    agent=hallucination_grader,
    context=[grader_task],
)

Ten kod definiuje zadanie o nazwie `hallucination_task`, którego celem jest ocena, czy odpowiedź nie zawiera halucynacji – czyli czy jest oparta na faktach i zgodna z pytaniem.

*   `hallucination_task = Task(...)`: Tworzy nowy obiekt o nazwie `hallucination_task` będący instancją klasy `Task`.
*   `description=(...)`: Definiuje opis zadania, który instruuje agenta, aby ocenił, czy odpowiedź jest oparta na faktach i wspierana przez dowody.
*   `expected_output=(...)`: Definiuje oczekiwany wynik zadania, czyli binarną ocenę "yes" (tak) lub "no" (nie), wskazującą na brak halucynacji. Agent ma odpowiedzieć "yes", jeśli odpowiedź jest użyteczna i zawiera fakty dotyczące pytania, a "no" w przeciwnym przypadku. Wynik ma być ograniczony wyłącznie do słów "yes" lub "no".
*   `agent=hallucination_grader`: Przypisuje agenta `hallucination_grader` do tego zadania. Agent ten będzie odpowiedzialny za wykonanie zadania zgodnie z podanym opisem.
*   `context=[grader_task]`: Określa kontekst zadania, czyli listę zadań, których wyniki są dostępne dla agenta podczas wykonywania tego zadania. W tym przypadku jedynym elementem w kontekście jest `grader_task`, co oznacza, że agent `hallucination_grader` będzie miał dostęp do wyniku wykonania zadania oceny trafności (`Grader_agent`).

W rezultacie powstaje zadanie `hallucination_task`, które ma na celu ocenę, czy odpowiedź nie zawiera halucynacji i wykorzystuje agenta `hallucination_grader`.

In [25]:

answer_task = Task(
    description=("Based on the response from the hallucination task for the quetion {question} evaluate whether the answer is useful to resolve the question."
    "If the answer is 'yes' return a clear and concise answer."
    "If the answer is 'no' then perform a 'websearch' and return the response"),
    expected_output=("Return a clear and concise response if the response from 'hallucination_task' is 'yes'."
    "Perform a web search using 'web_search_tool' and return ta clear and concise response only if the response from 'hallucination_task' is 'no'."
    "Otherwise respond as 'Sorry! unable to find a valid response'."),
    context=[hallucination_task],
    agent=answer_grader,
)

Ten kod definiuje zadanie o nazwie `answer_task`, którego celem jest sformułowanie końcowej odpowiedzi na pytanie użytkownika, uwzględniając wyniki wcześniejszych etapów oceny.

*   `answer_task = Task(...)`: Tworzy nowy obiekt o nazwie `answer_task` będący instancją klasy `Task`.
*   `description=(...)`: Definiuje opis zadania, który instruuje agenta, aby na podstawie wyniku z zadania `hallucination_task` ocenił przydatność odpowiedzi. Jeśli wynik jest "yes", agent ma zwrócić jasną i zwięzłą odpowiedź. Jeśli wynik jest "no", agent ma przeprowadzić wyszukiwanie w internecie za pomocą narzędzia `websearch` i zwrócić nową, jasną i zwięzłą odpowiedź.
*   `expected_output=(...)`: Definiuje oczekiwany wynik zadania: jeśli wynik z `hallucination_task` to "yes", agent ma zwrócić jasną i zwięzłą odpowiedź; jeśli wynik to "no", agent ma przeprowadzić wyszukiwanie w internecie i zwrócić nową, jasną i zwięzłą odpowiedź. W przeciwnym razie (jeśli wystąpi błąd lub nie można znaleźć odpowiedzi) agent ma odpowiedzieć "Sorry! unable to find a valid response".
*   `context=[hallucination_task]`: Określa kontekst zadania, czyli listę zadań, których wyniki są dostępne dla agenta podczas wykonywania tego zadania. W tym przypadku jedynym elementem w kontekście jest `hallucination_task`, co oznacza, że agent `answer_grader` będzie miał dostęp do wyniku wykonania zadania oceny halucynacji.
*   `agent=answer_grader`: Przypisuje agenta `answer_grader` do tego zadania. Agent ten będzie odpowiedzialny za wykonanie zadania zgodnie z podanym opisem.

W rezultacie powstaje zadanie `answer_task`, które ma na celu sformułowanie końcowej odpowiedzi, uwzględniając wyniki wcześniejszych etapów oceny i wykorzystuje agenta `answer_grader`.

# Załoga

In [26]:
rag_crew = Crew(
    agents=[Router_Agent, Retriever_Agent, Grader_agent, hallucination_grader, answer_grader],
    tasks=[router_task, retriever_task, grader_task, hallucination_task, answer_task],
    verbose=True,

)

Ten kod tworzy zespół agentów (Crew) o nazwie `rag_crew`, który będzie współpracował w celu udzielenia odpowiedzi na pytanie użytkownika.

*   `rag_crew = Crew(...)`: Tworzy nowy obiekt o nazwie `rag_crew` będący instancją klasy `Crew`.
*   `agents=[Router_Agent, Retriever_Agent, Grader_agent, hallucination_grader, answer_grader]`: Określa listę agentów wchodzących w skład zespołu. Kolejność agentów na liście definiuje kolejność ich działania. Są to: agent routingu (`Router_Agent`), agent pobierający informacje (`Retriever_Agent`), agent oceniający trafność (`Grader_agent`), agent oceniający halucynacje (`hallucination_grader`) i agent formułujący odpowiedź (`answer_grader`).
*   `tasks=[router_task, retriever_task, grader_task, hallucination_task, answer_task]`: Określa listę zadań, które mają być wykonane przez zespół. Kolejność zadań na liście definiuje kolejność ich wykonywania. Każde zadanie jest przypisane do konkretnego agenta.
*   `verbose=True`: Włącza tryb szczegółowy, co oznacza, że będą wyświetlane informacje o działaniach zespołu i poszczególnych agentów.

W rezultacie powstaje zespół `rag_crew`, który jest gotowy do pracy nad udzieleniem odpowiedzi na pytanie użytkownika poprzez sekwencyjne wykonywanie zadań przez odpowiednich agentów. Zespół ten wykorzystuje technikę RAG (Retrieval-Augmented Generation) – pobiera informacje z różnych źródeł, ocenia ich trafność i generuje odpowiedź opartą na tych informacjach.

In [27]:
inputs = {"question": "Tell me about self-attention mechanism in Transformers?"}
result = rag_crew.kickoff(inputs=inputs)


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 14fb2359-4057-4646-b909-5000043623b0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 6cb1dd88-792a-435a-84a6-58906208a4dc
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 6cb1dd88-792a-435a-84a6-58906208a4dc
       Status: Executing Task...
    └── 🤖 Agent: Router
            Status: In Progress

# Agent: Router
## Task: Analyse the keywords in the question Tell me about self-attention mechanism in Transformers?Based on the keywords decide whether it is eligible for a vectorstore search or a web search.Return a single word 'vectorstore' if it is eligible for vectorstore search.Return a single word 'websearch' if it is eligible for web search.Do not provide any other premable or explaination.


🤖 Agent: Router
    Status: In Progress
└── 🧠 Thinking...

...

# Agent: Answer Grader
## Task: Based on the response from the hallucination task for the quetion Tell me about self-attention mechanism in Transformers? evaluate whether the answer is useful to resolve the question.If the answer is 'yes' return a clear and concise answer.If the answer is 'no' then perform a 'websearch' and return the response


# Agent: Answer Grader
## Final Answer: 
The self-attention mechanism in Transformers allows the model to weigh the importance of different words in a sentence when encoding them. It computes a set of attention scores that determine how much focus to place on other words in the input sequence for each word being processed. This is achieved through three main components: queries, keys, and values. Each word is transformed into these three vectors, and the attention scores are calculated by taking the dot product of the query with all keys, followed by a softmax operation to normalize the scores. The output is a weighted sum of the values, allowi

🚀 Crew: crew
├── 📋 Task: 6cb1dd88-792a-435a-84a6-58906208a4dc
│      Assigned to: Router
│      Status: ✅ Completed
│   └── 🤖 Agent: Router
│           Status: ✅ Completed
├── 📋 Task: 26d34c08-b875-42ee-902c-b9ba4dc823c2
│      Assigned to: Retriever
│      Status: ✅ Completed
│   └── 🤖 Agent: Retriever
│           Status: ✅ Completed
├── 📋 Task: e07bb0f9-4e81-4830-b34e-db9cdceb6e30
│      Assigned to: Answer Grader
│      Status: ✅ Completed
│   └── 🤖 Agent: Answer Grader
│           Status: ✅ Completed
├── 📋 Task: a7bb2581-a3c3-4130-bbb5-9a650d0709fd
│      Assigned to: Hallucination Grader
│      Status: ✅ Completed
│   └── 🤖 Agent: Hallucination Grader
│           Status: ✅ Completed
└── 📋 Task: 6545f4ea-e2e2-4f3c-87e5-634b57ac9346
       Status: Executing Task...
    └── 🤖 Agent: Answer Grader
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 6cb1dd88-792a-435a-84a6-58906208a4dc
│      Assigned to: Router
│      Status: ✅ Completed
│   └── 🤖 Agent: Router
│           Status: ✅ Completed
├── 📋 Task: 26d34c08-b875-42ee-902c-b9ba4dc823c2
│      Assigned to: Retriever
│      Status: ✅ Completed
│   └── 🤖 Agent: Retriever
│           Status: ✅ Completed
├── 📋 Task: e07bb0f9-4e81-4830-b34e-db9cdceb6e30
│      Assigned to: Answer Grader
│      Status: ✅ Completed
│   └── 🤖 Agent: Answer Grader
│           Status: ✅ Completed
├── 📋 Task: a7bb2581-a3c3-4130-bbb5-9a650d0709fd
│      Assigned to: Hallucination Grader
│      Status: ✅ Completed
│   └── 🤖 Agent: Hallucination Grader
│           Status: ✅ Completed
└── 📋 Task: 6545f4ea-e2e2-4f3c-87e5-634b57ac9346
       Assigned to: Answer Grader
       Status: ✅ Completed
    └── 🤖 Agent: Answer Grader
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 6545f4ea-e2e2-4f3c-87e5-634b57ac9346                                                                     │
│  Agent: Answer Grader                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 14fb2359-4057-4646-b909-5000043623b0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [28]:
print(result)

The self-attention mechanism in Transformers allows the model to weigh the importance of different words in a sentence when encoding them. It computes a set of attention scores that determine how much focus to place on other words in the input sequence for each word being processed. This is achieved through three main components: queries, keys, and values. Each word is transformed into these three vectors, and the attention scores are calculated by taking the dot product of the query with all keys, followed by a softmax operation to normalize the scores. The output is a weighted sum of the values, allowing the model to capture contextual relationships and dependencies between words effectively. This mechanism is crucial for tasks such as translation, summarization, and other natural language processing applications.
